In [ ]:
# Supplementary Figure 16: synapse and matched neuronal cable proximity by neurotransmitter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

matched_file = Path('data/matched_synapse_cable.csv.gz')
out_dir = Path('supp_outputs')
out_dir.mkdir(parents=True, exist_ok=True)
nt_order = ['ACH', 'GABA', 'GLUT', 'DA', 'SER', 'OCT']
nt_colors = {'ACH': '#2020df', 'GABA': '#df2020', 'GLUT': '#df9c20',
             'DA': '#963939', 'OCT': '#107010', 'SER': '#701070'}

matched = pd.read_csv(matched_file, dtype={'pre_root_id': 'string'}, low_memory=False)
matched = matched[~matched['neuropil'].isin(['LA_L', 'LA_R'])]
matched['nt_type'] = matched['nt_type'].str.strip().str.upper().replace({
    'ACETYLCHOLINE': 'ACH', 'GLUTAMATE': 'GLUT', 'DOPAMINE': 'DA',
    'SEROTONIN': 'SER', '5HT': 'SER', '5-HT': 'SER', 'OCTOPAMINE': 'OCT'})
matched = matched.sort_values(['match_id', 'is_synapse'], ascending=[True, False], kind='stable')
distance4 = matched['distance_to_trachea_surface_um'].to_numpy(float).reshape(-1, 4)
nt_labels = matched['nt_type'].to_numpy()[::4]
grid = np.linspace(0, 40, 1601)

def ecdf(values, grid):
    values = np.sort(values)
    return np.searchsorted(values, grid, side='right') / len(values)

plt.rcParams.update({'font.family': 'Arial', 'font.size': 10, 'svg.fonttype': 'none'})
fig, axes = plt.subplots(3, 2, figsize=(9.2, 10.2))
for ax, nt in zip(axes.ravel(), nt_order):
    group = distance4[nt_labels == nt]
    syn = group[:, 0]
    cable = group[:, 1:].reshape(-1)
    ax.plot(grid, ecdf(syn, grid), color=nt_colors[nt], lw=1.7, label='Presynaptic sites')
    ax.plot(grid, ecdf(cable, grid), color='0.4', lw=1.7, label='Matched cable')
    ax.set(xlim=(0, 40), ylim=(0, 1), title=nt,
           xlabel='Distance to tracheole surface (µm)', ylabel='Cumulative fraction')
    ax.spines[['top', 'right']].set_visible(False)
axes[0, 0].legend(frameon=False, loc='lower right')
fig.tight_layout()
fig.savefig(out_dir / 'supp_fig16.svg', bbox_inches='tight')
plt.show()